# 04 · Create the Direct Lake semantic model

Stitches the gold star schema to a **Power BI semantic model in Direct Lake mode**, driven entirely by
`model_spec.json`. Uses [Semantic Link Labs](https://github.com/microsoft/semantic-link-labs) (`sempy_labs`),
which wraps the Tabular Object Model (TOM) so the model can be created and shaped from a notebook:

1. Creates (or overwrites) a Direct Lake model containing every gold table.
2. Adds relationships: fact → dimension on surrogate keys, fact → `dim_date` for every date key
   (first one active, the rest inactive for `USERELATIONSHIP`), and marks `dim_date` as the date table.
3. Hides key columns, switches keys' default summarisation off, sets sort-by columns for month / day names,
   adds a `Year > Quarter > Month` hierarchy, and creates explicit measures with format strings.
4. Refreshes (reframes) the model and runs a smoke-test DAX query.
5. Exports a `.bim` (Tabular Model Scripting Language) copy to `Files/model/` for source control, Tabular Editor,
   or deployment to another workspace with `labs.create_semantic_model_from_bim`.

**Requirements:** run inside Fabric (the notebook's identity needs Contributor on the workspace); the Lakehouse
must have a SQL analytics endpoint (all Lakehouses do); `semantic-link-labs` is installed in the first cell.

In [ ]:
%pip install -q "semantic-link-labs>=0.17"

In [ ]:
# PARAMETERS — override from a pipeline or notebookutils.notebook.run()
LAKEHOUSE_ROOT = ""            # abfss://<workspace-id>@onelake.dfs.fabric.microsoft.com/<lakehouse-id> ; "" = default lakehouse
SPEC_PATH = "Files/model/model_spec.json"
SEMANTIC_MODEL_NAME = ""       # "" = spec's model_name
TARGET_WORKSPACE = ""          # workspace name or id for the semantic model; "" = this notebook's workspace
LAKEHOUSE_NAME = ""            # lakehouse the gold tables live in; "" = default lakehouse
LAKEHOUSE_WORKSPACE = ""       # workspace of that lakehouse; "" = TARGET_WORKSPACE
GOLD_SCHEMA = ""               # "" = spec's gold_schema
FRIENDLY_TABLE_NAMES = True    # dim_customer → "Customer", fact_order_items → "Order Items", dim_date → "Date"
USE_SQL_ENDPOINT = False       # False = Direct Lake over OneLake (recommended); True = Direct Lake over the SQL analytics endpoint
OVERWRITE_EXISTING = True      # drop and recreate the model if it already exists
REFRESH_AFTER_BUILD = True
EXPORT_BIM = True

In [ ]:
import json, re, datetime as dt
import sempy.fabric as fabric
import sempy_labs as labs
from sempy_labs import directlake as dl
from sempy_labs.tom import connect_semantic_model


def _lakehouse_root():
    if LAKEHOUSE_ROOT:
        return LAKEHOUSE_ROOT.rstrip("/")
    ctx = notebookutils.runtime.context
    ws, lh = ctx.get("defaultLakehouseWorkspaceId"), ctx.get("defaultLakehouseId")
    if not lh:
        raise RuntimeError("Attach a default Lakehouse to this notebook or set LAKEHOUSE_ROOT.")
    return f"abfss://{ws}@onelake.dfs.fabric.microsoft.com/{lh}"


ROOT = _lakehouse_root()
spec = json.loads(notebookutils.fs.head(f"{ROOT}/{SPEC_PATH}", 50 * 1024 * 1024))
MODEL = SEMANTIC_MODEL_NAME or spec["model_name"]
GOLD = GOLD_SCHEMA or spec.get("gold_schema", "gold")
SCHEMA_ENABLED = bool(spec.get("schema_enabled", True))
UNKNOWN = int(spec.get("unknown_member_key", -1))

ctx = notebookutils.runtime.context
workspace = TARGET_WORKSPACE or ctx.get("currentWorkspaceId")
lakehouse = LAKEHOUSE_NAME or ctx.get("defaultLakehouseName") or labs.resolve_lakehouse_name()
lakehouse_workspace = LAKEHOUSE_WORKSPACE or ctx.get("defaultLakehouseWorkspaceId") or workspace
workspace_name = fabric.resolve_workspace_name(workspace)

DATE_DIM = spec["date_dimension"]["name"]
dims = {d["name"]: d for d in spec["dimensions"]}
facts = spec["facts"] + spec.get("bridges", [])


def _title(s):
    return " ".join(w.capitalize() for w in re.split(r"[_\s]+", s) if w)


def gold_source(name):
    """Schema-qualified lakehouse table name for the generator."""
    return f"{GOLD}.{name}" if SCHEMA_ENABLED else f"{GOLD}_{name}"


def model_table_name(gold_name):
    """Name of the table inside the semantic model."""
    if not FRIENDLY_TABLE_NAMES:
        return gold_name if SCHEMA_ENABLED else f"{GOLD}_{gold_name}"
    return _title(re.sub(r"^(dim_|fact_|bridge_)", "", gold_name))


gold_names = [DATE_DIM] + list(dims) + [f["name"] for f in facts]
tables_map = {model_table_name(n): gold_source(n) for n in gold_names}
if len(tables_map) != len(gold_names):
    raise RuntimeError("Friendly table names collide (e.g. dim_order and fact_order) — set FRIENDLY_TABLE_NAMES = False.")
print(f"Semantic model: {MODEL}\nWorkspace:      {workspace_name}\nLakehouse:      {lakehouse}\nTables:")
for k, v in tables_map.items():
    print(f"  {k:30s} ← {v}")

## Step 1 — Create the Direct Lake model from the gold tables
`generate_direct_lake_semantic_model` creates the model, the shared Direct Lake expression pointing at the Lakehouse's
SQL analytics endpoint, and one table per gold table with every column mapped.

In [ ]:
existing = fabric.list_datasets(workspace=workspace)
if MODEL in existing["Dataset Name"].values and not OVERWRITE_EXISTING:
    raise RuntimeError(f"Semantic model '{MODEL}' already exists in {workspace_name}. Set OVERWRITE_EXISTING = True to replace it.")

dl.generate_direct_lake_semantic_model(
    dataset=MODEL,
    tables=tables_map,                 # {model table name: "schema.lakehouse_table"}
    source=lakehouse,
    source_type="Lakehouse",
    source_workspace=lakehouse_workspace,
    use_sql_endpoint=USE_SQL_ENDPOINT,
    workspace=workspace,
    refresh=False,
    overwrite=OVERWRITE_EXISTING,
)
print(f"Created Direct Lake model '{MODEL}' with {len(tables_map)} tables")

## Step 2 — Relationships, date table, key hygiene, hierarchies, measures
Everything below is idempotent: it looks up what exists before adding.

In [ ]:
relationships_added, measures_added = [], []
with connect_semantic_model(dataset=MODEL, readonly=False, workspace=workspace) as tom:
    tables = {t.Name: t for t in tom.model.Tables}
    have_rel = {(r.FromTable.Name, r.FromColumn.Name, r.ToTable.Name, r.ToColumn.Name) for r in tom.model.Relationships}

    def add_rel(from_table, from_col, to_table, to_col, active=True):
        key = (from_table, from_col, to_table, to_col)
        if key in have_rel:
            return
        if from_table not in tables or to_table not in tables:
            print(f"  skip relationship {key}: table missing in model"); return
        tom.add_relationship(from_table=from_table, from_column=from_col, to_table=to_table, to_column=to_col,
                             from_cardinality="Many", to_cardinality="One", cross_filtering_behavior="OneDirection",
                             is_active=active, security_filtering_behavior="OneDirection", rely_on_referential_integrity=False)
        have_rel.add(key); relationships_added.append(key + (active,))

    # --- date table ---
    dt_name = model_table_name(DATE_DIM)
    if dt_name in tables:
        tom.mark_as_date_table(table_name=dt_name, column_name="full_date")
        for c in ("date_key", "year_month_key", "day_of_week", "day_of_year", "week_of_year", "fiscal_period"):
            if c in [col.Name for col in tables[dt_name].Columns]:
                tom.set_summarize_by(table_name=dt_name, column_name=c, value="None")
        tables[dt_name].Columns["date_key"].IsHidden = True
        tom.set_sort_by_column(table_name=dt_name, column_name="month_name", sort_by_column="month")
        tom.set_sort_by_column(table_name=dt_name, column_name="month_short", sort_by_column="month")
        tom.set_sort_by_column(table_name=dt_name, column_name="day_name", sort_by_column="day_of_week")
        tom.set_sort_by_column(table_name=dt_name, column_name="quarter_name", sort_by_column="year_month_key")
        if "Calendar" not in [h.Name for h in tables[dt_name].Hierarchies]:
            tom.add_hierarchy(table_name=dt_name, hierarchy_name="Calendar", columns=["year", "quarter_name", "month_name", "full_date"],
                              levels=["Year", "Quarter", "Month", "Date"])
        if "Fiscal" not in [h.Name for h in tables[dt_name].Hierarchies]:
            tom.add_hierarchy(table_name=dt_name, hierarchy_name="Fiscal", columns=["fiscal_year", "fiscal_quarter", "fiscal_period", "full_date"],
                              levels=["Fiscal Year", "Fiscal Quarter", "Fiscal Period", "Date"])

    # --- dimensions: hide surrogate key, no summarisation on keys ---
    for d in dims.values():
        tn = model_table_name(d["name"])
        if tn not in tables:
            continue
        cols = {c.Name: c for c in tables[tn].Columns}
        if d["surrogate_key"] in cols:
            cols[d["surrogate_key"]].IsHidden = True
            tom.set_summarize_by(table_name=tn, column_name=d["surrogate_key"], value="None")
        for nk in d["natural_key"]:
            if nk in cols:
                tom.set_summarize_by(table_name=tn, column_name=nk, value="None")
        if "_etl_loaded_at" in cols:
            cols["_etl_loaded_at"].IsHidden = True

    # --- facts: relationships, hide keys, measures ---
    for f in facts:
        tn = model_table_name(f["name"])
        if tn not in tables:
            print(f"  skip {tn}: not in model"); continue
        cols = {c.Name: c for c in tables[tn].Columns}
        for fk in f["foreign_keys"]:
            dn = model_table_name(fk["dimension"])
            add_rel(tn, fk["surrogate_key_column"], dn, dims[fk["dimension"]]["surrogate_key"], active=True)
            if fk["surrogate_key_column"] in cols:
                cols[fk["surrogate_key_column"]].IsHidden = True
                tom.set_summarize_by(table_name=tn, column_name=fk["surrogate_key_column"], value="None")
        for dc in f["date_columns"]:
            add_rel(tn, dc["date_key_column"], dt_name, "date_key", active=bool(dc["active"]))
            if dc["date_key_column"] in cols:
                cols[dc["date_key_column"]].IsHidden = True
                tom.set_summarize_by(table_name=tn, column_name=dc["date_key_column"], value="None")
        if "_etl_loaded_at" in cols:
            cols["_etl_loaded_at"].IsHidden = True
        for dg in f["degenerate_columns"]:
            if dg in cols and str(cols[dg].DataType) in ("Int64", "Double", "Decimal"):
                tom.set_summarize_by(table_name=tn, column_name=dg, value="None")
        existing_measures = {m.Name for m in tables[tn].Measures}
        folder = _title(f["name"].replace("fact_", "").replace("bridge_", ""))
        for m in f["measures"]:
            if m["measure_name"] in existing_measures or m["column"] not in cols:
                continue
            tom.add_measure(table_name=tn, measure_name=m["measure_name"], expression=f"SUM('{tn}'[{m['column']}])",
                            format_string=m["format_string"], display_folder=folder)
            cols[m["column"]].IsHidden = True  # explicit measures replace implicit column aggregation
            measures_added.append(m["measure_name"])
        if f["row_count_measure"] not in existing_measures:
            tom.add_measure(table_name=tn, measure_name=f["row_count_measure"], expression=f"COUNTROWS('{tn}')", format_string="#,##0", display_folder=folder)
            measures_added.append(f["row_count_measure"])
        # inactive date relationships get a ready-made USERELATIONSHIP measure for the first sum measure
        first_sum = next((m for m in f["measures"] if m["column"] in cols), None)
        for dc in f["date_columns"]:
            if not dc["active"] and first_sum:
                name = f"{first_sum['measure_name']} by {_title(dc['role'])} Date"
                if name not in existing_measures:
                    tom.add_measure(table_name=tn, measure_name=name, display_folder=folder, format_string=first_sum["format_string"],
                                    expression=f"CALCULATE([{first_sum['measure_name']}], USERELATIONSHIP('{tn}'[{dc['date_key_column']}], '{dt_name}'[date_key]))")
                    measures_added.append(name)

print(f"Relationships added: {len(relationships_added)}")
for r in relationships_added:
    print(f"  {r[0]}[{r[1]}] → {r[2]}[{r[3]}]" + ("" if r[4] else "  (inactive)"))
print(f"Measures added: {len(measures_added)}")
for m in measures_added:
    print("  ", m)

## Step 3 — Refresh (reframe) and smoke-test
A Direct Lake refresh only re-reads Delta metadata; it is fast. The DAX query proves the relationships filter correctly.

In [ ]:
if REFRESH_AFTER_BUILD:
    labs.refresh_semantic_model(dataset=MODEL, workspace=workspace, refresh_type="full")
    print("Refreshed", MODEL)

display(fabric.list_relationships(dataset=MODEL, workspace=workspace))

first_fact = facts[0]
ft = model_table_name(first_fact["name"])
measure = first_fact["measures"][0]["measure_name"] if first_fact["measures"] else first_fact["row_count_measure"]
dax = f"""
EVALUATE
TOPN(12, SUMMARIZECOLUMNS('{model_table_name(DATE_DIM)}'[year], '{model_table_name(DATE_DIM)}'[month_name], "Value", [{measure}]), [Value], DESC)
"""
print(dax)
display(fabric.evaluate_dax(dataset=MODEL, dax_string=dax, workspace=workspace))

## Step 4 — Export a `.bim` for source control / redeployment
The exported TMSL definition can be opened in Tabular Editor, committed to Git alongside `model_spec.json`, or redeployed with
`labs.create_semantic_model_from_bim(dataset, bim_file, workspace)`.

In [ ]:
if EXPORT_BIM:
    bim = labs.get_semantic_model_bim(dataset=MODEL, workspace=workspace)
    out = f"{ROOT}/{SPEC_PATH.rsplit('/', 1)[0]}/{re.sub(r'[^A-Za-z0-9_-]+', '_', MODEL)}.bim"
    notebookutils.fs.put(out, json.dumps(bim, indent=2), True)
    print("Exported", out)

print(f"\nDone. Open the semantic model '{MODEL}' in workspace '{workspace_name}' and start building reports, "
      "or run labs.model_bpa(dataset=MODEL, workspace=workspace) for best-practice-analyzer checks.")